In [ ]:
import tables
import numpy as np
import matplotlib.pyplot as plt

#a=tables.open_file("/data/cta/users-ifae/summer_students/agl/Notebooks/zscores_1506.h5")
a=tables.open_file("/data/cta/users-ifae/summer_students/agl/scripts/zscores.h5")

In [ ]:
a.root

In [ ]:
a.root.zscores

In [ ]:
a.root.invalid_runs

In [ ]:
#let's see how many of them failed filter 1 or 2
invalid = a.root.invalid_runs[:]

print("Filtro 1:", np.sum(invalid["filter_failed"] == 1))
print( (invalid [invalid["filter_failed"] == 1])["run_id"])

print("Filtro 2:", np.sum(invalid["filter_failed"] == 2))
print(invalid[invalid["filter_failed"] == 2]["run_id"])


# F1 is not appliable only to 41/1500 Runs
# F1 is not appliable only to 51/1500 Runs

In [ ]:
# check for which subruns both of the filter failed so to not run toy2 in them
invalid = a.root.invalid_runs.read()

runs = invalid["run_id"]
filters = invalid["filter_failed"]

failed_both = []

for run in np.unique(runs):
    f = filters[runs == run]
    if 1 in f and 2 in f:
        failed_both.append(run)

print(failed_both)
print(len(failed_both))

In [ ]:
# check to locate where did sigma2==0 appeared
z1 = a.root.zscores.col("z_score_1")
z2 = a.root.zscores.col("z_score_2")

print(np.isinf(z2).sum())

idx = np.where(np.isinf(z2))[0] # z2 are tuples (idx, value) look for 
print(a.root.zscores.col('run_id')[idx])

In [ ]:
# QUICK PLOT OF THE Z SCORES 

plt.figure(figsize=(10, 6))
plt.hist(z1, bins=150, color='plum',  edgecolor='black', linewidth=0.4) #matplotlib ignores the nans
plt.title('z_score_1 distribuition for ~1500 runs')
plt.xlabel('z_score_1')
#plt.xlim(-10,150)
plt.yscale('log')
#plt.xscale('log')
#plt.ylim(0,3000)
#plt.xticks(np.arange(0, 6500, step=500)) 
plt.show()


plt.figure(figsize=(10, 6))
plt.hist(z2[np.isfinite(z2)], bins=150, color='mediumpurple',  edgecolor='black', linewidth=0.4)
plt.title('z_score_2 distribuition for ~1500 runs')
plt.xlabel('z_score_2')
#plt.xlim(-10,150)
plt.yscale('log')
#plt.xscale('log')
#plt.ylim(0,3000)
#plt.xticks(np.arange(0, 425, step=25)) 
plt.show()


In [ ]:
# PLOT OF NUMBER OF SIGMAS VERSUS THE RATIO OF SURVIVING POINTS
n_sigmas = np.arange(1, 200, 1)

anomalous_1 = []
anomalous_2 = []

for n in n_sigmas:
    anomalous_1.append(np.sum(z1 > n) / len(z1))
    anomalous_2.append(np.sum(z2 > n) / len(z2))


sigma_cutoff_1 = 120
sigma_cutoff_2 = 15

plt.figure(figsize=(10, 6))
plt.plot(n_sigmas, anomalous_1, label="Filter 1", linewidth=1)
plt.plot(n_sigmas, anomalous_2, label="Filter 2", linewidth=1)
plt.axhline(y=0.01, color='red', linestyle='--', linewidth=1) # to find the sigma at which the percentage is 1/1000
#plt.xticks(np.arange(0, 100, step=10)) 
plt.yticks(np.arange(0, 1, step=0.1))
plt.xlabel(r"# of $\sigma$")
plt.ylabel(r"# of subruns with z-score > $\sigma$ / # of subruns")
plt.yscale("log")      # opcional pero muy recomendable
plt.axvline(x= sigma_cutoff_1, color='black', linestyle='--', label=f'Sigma cutoff {sigma_cutoff_1}')
plt.axvline(x= sigma_cutoff_2, color='black', linestyle='--', label=f'Sigma cutoff {sigma_cutoff_2}')
plt.legend()
plt.grid(True)
plt.show()


# Filter 1----------------------------------------------------------------------------------
# stablishing a threshold 
mask_1 = z1 > sigma_cutoff_1  
strange_runs = a.root.zscores.col("run_id")[mask_1]
#strange_subruns = a.root.zscores.col("subrun_index")[mask]
print(len(strange_runs))

plt.figure(figsize=(10, 6))
plt.hist(z1, bins=400, color='plum',  edgecolor='black', linewidth=0.4)
plt.title('z_score_1 distribuition for ~1500 runs')
plt.xlabel('z_score_1')
#plt.xscale('log')
plt.xlim(0,6000)
plt.yscale('log')
plt.axvline(x= sigma_cutoff_1, color='black', linestyle='--', label=f'Sigma cutoff {sigma_cutoff_1}')
#plt.xticks(np.arange(0, 6500, step=500)) 
plt.legend()
plt.show()


# Filter 2----------------------------------------------------------------------------------------
#stablishing a thershold

mask_2 = z2 > sigma_cutoff_2  
strange_runs = a.root.zscores.col("run_id")[mask_2]
#strange_subruns = a.root.zscores.col("subrun_index")[mask]
print(len(strange_runs))

plt.figure(figsize=(10, 6))
plt.hist(z2[np.isfinite(z2)], bins=150, color='mediumpurple',  edgecolor='black', linewidth=0.4)
plt.title('z_score_2 distribuition for ~1500 runs')
plt.xlabel('z_score_2')
plt.yscale('log')
#plt.xscale('log')
plt.axvline(x= sigma_cutoff_2, color='black', linestyle='--', label=f'Sigma cutoff {sigma_cutoff_2}')
#plt.xticks(np.arange(0, 425, step=25)) 
plt.legend()
plt.show()







#hist bidimensional. sigmas vs zs 

In [ ]:
mask = np.isfinite(z1) & np.isfinite(z2) 
# isfinite = False if z1 nan and True if it's a number
# a & b takes the runs such that z1 and z2 are not nans

plt.figure(figsize=(7,6))
plt.hist2d(z1[mask], z2[mask], bins=150)
plt.colorbar(label="Number of subruns")
plt.xscale("log")
plt.yscale("log")

plt.xlabel("z_score_1")
plt.ylabel("z_score_2")
plt.show()